2025_05 【必須スキル】クレンジング

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
print(os.getcwd())

/app/src


In [3]:
# データの読み込み
# クレンジング対象の求人情報データ（id・企業名・スキル）
jobs_df = pd.read_csv('../data/cleansing_data.csv')

# スキルマッピング表
mapping_df = pd.read_csv('../data/key_mappingu.csv')

In [4]:
# データの確認
print(jobs_df.head(10))

   id       company_name                                    required_skills
0   0        株式会社シンフォニード  求める人材: \n＜必須条件＞\n下記項目を作業及びPMした経験（いずれも1年以上）\n\n...
1   1   Indeed Japan株式会社                                                NaN
2   2        株式会社シンフォニード  求める人材: \n＜必須条件＞\nデータ分析（アプリまたはWEBサービス）\n・SQLを用い...
3   3        株式会社ヘルスベイシス  求める人材: \n＜スキル＞\n・Pythonの実務利用経験\n・SQLの実務利用経験\n求...
4   4        株式会社シンフォニード  求める人材: \n＜必須条件＞\n?購買データや会員データ、Web行動データなどを用いたデー...
5   5  株式会社クリーク・アンド・リバー社  求める人材: \n【応募要件（MUST）】\n▼下記のすべてのご経験・スキルをお持ちの方\n...
6   6          株式会社エイジレス  求める人材: \n＜必須要件＞\n・コミュニケーション能力および論理的思考\n・標準SQLで...
7   7        株式会社シンフォニード  求める人材: \n＜必須条件＞\n・データ分析によってビジネス上の課題を解決した経験（直近含...
8   8           バリュー株式会社  求める人材: \n＜以下のすべてを満たす方＞\n・データ／テクノロジーを駆使した企業変革、社...
9   9  株式会社クリーク・アンド・リバー社  求める人材: \n【必要要件】\n▼以下いずれかの経験を有する方\n・データマネジメントに関...


In [5]:
# データの確認
print(mapping_df.head(10))

   ID            raw_skill standard_skill
0   1                Adobe          Adobe
1   2              ANDROID        Android
2   3              Android        Android
3   4               APACHE         Apache
4   5               Apache         Apache
5   6                AVAYA          Avaya
6   7                Avaya          Avaya
7   8                  AWS            AWS
8   9  Amazon Web Services            AWS
9  10                AZURE          Azure


In [6]:
# マッチング結果の初期化
job_skill_pairs = []       # 中間テーブル用のデータ
unmatched_skills = set()   # マッチしなかったスキル文

# スキルの抽出に使用する正規表現
import re
skill_pattern = re.compile(r"・([^\n]+)")

In [11]:
# スキル項目のクレンジング
for idx, row in jobs_df.iterrows():
    job_id = row['id']
    company = row['company_name']

    # 改行や不要な空白を削除
    skill_text = str(row['required_skills']).replace('\n', ' ').strip().lower()

    # 正規表現でスキルを抽出
    skills = skill_pattern.findall(skill_text)

    # 抽出したスキルをマッチング処理
    matched_skills = set()

    for idx_mappling, mapping_row in mapping_df.iterrows():
        variant = str(mapping_row['raw_skill']).lower() # 小文字化
        normalized = mapping_row['standard_skill'] # 正規化されたスキル

        for skill in skills:
            if variant in skill:
                matched_skills.add(normalized)
    
    # マッチしたスキルがあれば、job_skill_parisに追加
    if matched_skills:
        for skill in matched_skills:
            job_skill_pairs.append({
                'job_id': job_id,
                'company_name': company,
                'skill': skill
            })
    else:
        unmatched_skills.add(skill_text)

In [ ]:
# クレンジングの確認(マッチOK)
for item in job_skill_pairs[:10]:
    print(item)

{'job_id': 0, 'company_name': '株式会社シンフォニード', 'skill': 'Python'}
{'job_id': 0, 'company_name': '株式会社シンフォニード', 'skill': 'R'}
{'job_id': 0, 'company_name': '株式会社シンフォニード', 'skill': 'SQL'}
{'job_id': 2, 'company_name': '株式会社シンフォニード', 'skill': 'Python'}
{'job_id': 2, 'company_name': '株式会社シンフォニード', 'skill': 'R'}
{'job_id': 2, 'company_name': '株式会社シンフォニード', 'skill': 'SQL'}
{'job_id': 3, 'company_name': '株式会社ヘルスベイシス', 'skill': 'Python'}
{'job_id': 3, 'company_name': '株式会社ヘルスベイシス', 'skill': 'SQL'}
{'job_id': 4, 'company_name': '株式会社シンフォニード', 'skill': 'Tableau'}
{'job_id': 4, 'company_name': '株式会社シンフォニード', 'skill': 'BIツール'}


In [18]:
# クレンジングの確認(マッチNO)
for text in list(unmatched_skills)[:10]:
    print(text)

求める人材:  ＜必須条件＞ ・クラウドベースでの情報系システム（dwhやbi）の設計構築の経験及び資格 ・データモデリングの実務経験 ・製品リリースにおけるスタンダードなソースコード管理のスキルや経験 ・dmbokベースのデータマネジメント全般の基礎知識 ・データ活用の推進経験、若しくはデータサイエンティストなど関連チームとの協業をした経験  ＜歓迎要件＞ 求めている人材の情報量は適切ですか？ 十分 不足
求める人材:  【必須スキル・経験】 ・プロジェクトマネジメント経験 ・システム開発経験 ※機械学習未経験でも歓迎  【尚可スキル・経験】 ・受託開発などのクライアントワーク経験 ・データサイエンスプロジェクトの経験  【求める人物像】 ・チャレンジングなビジネス課題に挑戦していきたい方 ・各チームを牽引し、メンバーへのフィードバックや技術的なメンターを率先していきたい方 求めている人材の情報量は適切ですか？ 十分 不足
求める人材:  既卒・第二新卒歓迎！ 20代・30代が多く在籍中！  ★応募資格はございません！ ★未経験大歓迎！  ≪応募に必要なのは「やる気」のみ≫ 活気のある環境で確実にスキルアップが可能です！  ◎イチからスキルを身につけてプロのエンジニアを目指したい ◎好きなことを仕事にして自由に働きながら活躍したい ◎自分の人生をガラッと変えたい  そんなやる気や意欲に溢れている方は幅広く活躍できる環境です！ 求めている人材の情報量は適切ですか？ 十分 不足
求める人材:  【未経験でもフリーターでも大丈夫】 ・職歴不問 ・フリーター歓迎 ・社会人デビュー歓迎 ・ワークライフバランス充実 ・育成前提の充実研修 求めている人材の情報量は適切ですか？ 十分 不足
求める人材:  【学歴】 不問  【応募に必須な条件】 c/c++言語を用いたソフトウェア開発の実務経験を2年以上経験している方 linuxで動作するアプリケーションの開発経験を有する方  英文メールの読み書きが出来る方、英文の仕様書や技術文書の読解が出来る方  【歓迎される資格・経験】 業務において、高い意識を持って行動できる方 課題解決において、自身の経験と知見から仮説を立て、解決に導く力強さと勇気と決断力を備えている方 部内部外問わず、常に対等な目線でコミュニケーションが出来る方  【フ

In [19]:
# DBにインポートするため、DataFrameに変換

# job_skill_pairsをDataFrame化
job_skill_df = pd.DataFrame(job_skill_pairs)

# 重複排除して正規化スキルの一覧を作成
unique_skills = job_skill_df['skill'].drop_duplicates().reset_index(drop=True)

# skill_idを付与して、skillsテーブル（DataFrame）を作成する
skills_df = pd.DataFrame({
    'skill_id': unique_skills.index + 1,
    'keyword': unique_skills
})

In [23]:
# DataFreame化した後の確認
print("----- {} の最初の10行 -----".format("skills_df"))
print(skills_df.head(10))

print("\n----- {} の最初の10行 -----".format("job_skill_df"))
print(job_skill_df.head(10))

----- skills_df の最初の10行 -----
   skill_id   keyword
0         1    Python
1         2         R
2         3       SQL
3         4   Tableau
4         5     BIツール
5         6    Looker
6         7  BigQuery
7         8         C
8         9        Go
9        10       AWS

----- job_skill_df の最初の10行 -----
   job_id company_name    skill
0       0  株式会社シンフォニード   Python
1       0  株式会社シンフォニード        R
2       0  株式会社シンフォニード      SQL
3       2  株式会社シンフォニード   Python
4       2  株式会社シンフォニード        R
5       2  株式会社シンフォニード      SQL
6       3  株式会社ヘルスベイシス   Python
7       3  株式会社ヘルスベイシス      SQL
8       4  株式会社シンフォニード  Tableau
9       4  株式会社シンフォニード    BIツール


In [26]:
# csv出力
skills_df.to_csv("../data/skills.csv", index=False)
job_skill_df.to_csv("../data/job_posting_skills.csv", index=False)